# Practical 03: In-Memory Optimization via Dtype Downcasting & Deep Profiling

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to demonstrate in-memory optimization of large datasets. Profile baseline memory usage using memory_usage(deep=True), downcast integer columns (int64 -> int32/int16), downcast float columns (float64 -> float32), and convert repetitive string columns to category type. Report the total percentage reduction in RAM footprint.

In [1]:
import numpy as np
import pandas as pd

rs = np.random.default_rng(11)
n = 600_000
data = pd.DataFrame({
    "transaction_id": np.arange(1, n + 1, dtype="int64"),
    "customer_id": rs.integers(1, 80000, n),
    "store_id": rs.integers(1, 180, n),
    "quantity": rs.integers(1, 16, n),
    "amount": np.round(rs.gamma(2.5, 50.0, n), 2),
    "category": rs.choice(["Electronics", "Furniture", "Groceries", "Sports", "Books", "Toys"], n).astype(object),
    "payment_method": rs.choice(["Credit Card", "Debit Card", "UPI", "Cash", "Net Banking"], n).astype(object),
    "city": rs.choice(["Kochi", "Chennai", "Mumbai", "Bengaluru", "Hyderabad", "Pune", "Ahmedabad"], n).astype(object),
})
print(data.head())

def mb(frame):
    """Total deep memory of a DataFrame in MB."""
    return frame.memory_usage(deep=True).sum() / 1024 ** 2

base_mb = mb(data)
print(data.dtypes, "\n")
print("Memory per column (MB):")
print((data.memory_usage(deep=True) / 1024 ** 2).round(3))
print(f"\nBASELINE TOTAL: {base_mb:.2f} MB")

slim = data.copy()
for col in slim.select_dtypes(include="integer").columns:
    slim[col] = pd.to_numeric(slim[col], downcast="integer")

after_int = mb(slim)
print(slim.select_dtypes(include="integer").dtypes.to_dict())
print(f"After integer downcast: {after_int:.2f} MB")

for col in slim.select_dtypes(include="float").columns:
    slim[col] = pd.to_numeric(slim[col], downcast="float")

after_float = mb(slim)
print(slim.select_dtypes(include="float").dtypes.to_dict())
print(f"After float downcast: {after_float:.2f} MB")

for col in slim.select_dtypes(include=["object", "string"]).columns:
    if slim[col].nunique() / len(slim) < 0.4:
        slim[col] = slim[col].astype("category")

after_cat = mb(slim)
print(slim.dtypes)
print(f"\nAfter category conversion: {after_cat:.2f} MB")

reduction = (base_mb - after_cat) / base_mb * 100
print("=" * 45)
print(f"Baseline  : {base_mb:8.2f} MB")
print(f"Optimised : {after_cat:8.2f} MB")
print(f"REDUCTION : {reduction:8.2f} %")
print("=" * 45)

,transaction_id,customer_id,store_id,quantity,amount,category,payment_method,city
0,1,10704,167,8,102.85,Groceries,Cash,Kochi
1,2,10286,83,4,68.15,Furniture,UPI,Mumbai
2,3,63766,34,5,106.92,Books,Debit Card,Kochi
3,4,39942,79,11,159.38,Toys,UPI,Chennai
4,5,47203,95,13,153.32,Groceries,Cash,Mumbai


transaction_id      int64
customer_id         int64
store_id            int64
quantity            int64
amount            float64
category           object
payment_method     object
city               object
dtype: object 

Memory per column (MB):
Index              0.000
transaction_id     4.578
customer_id        4.578
store_id           4.578
quantity           4.578
amount             4.578
category          32.234
payment_method    32.499
city              32.044
dtype: float64

BASELINE TOTAL: 119.67 MB
{'transaction_id': dtype('int32'), 'customer_id': dtype('int32'), 'store_id': dtype('int16'), 'quantity': dtype('int8')}
After integer downcast: 107.65 MB
{'amount': dtype('float32')}
After float downcast: 105.36 MB
transaction_id       int32
customer_id          int32
store_id             int16
quantity              int8
amount             float32
category          category
payment_method    category
city              category
dtype: object

After category conversion: 10.30 MB
Ba